# Data Quality Notebook 01: Pandas reconciliation

This notebook introduces a practical data engineering task: reconciling a source table with a target table after a load or transformation.

In real work, reconciliation helps us answer: Did the data arrive? Did we lose records? Did we create duplicates? Did important values change?

## Notebook objectives

By the end of this notebook, you should be able to:

- explain what source-to-target reconciliation means
- compare row counts between two tables
- compare primary keys to find missing and extra records
- reconcile aggregate metrics such as total revenue
- compare matched records column by column
- create a simple reconciliation summary table

## 1. What reconciliation means

A common data pipeline loads data from a **source** into a **target**.

Examples:

- MySQL source table to Snowflake target table
- Raw file to cleaned warehouse table
- Redshift query output to a downstream reporting table
- Spark ETL output to a curated dataset

A reconciliation check compares both sides and gives us evidence that the pipeline worked correctly.

In [ ]:
import pandas as pd

## 2. Scenario: source orders and target orders

We will use a small sanitized order dataset. Imagine that `source_orders` is the original table and `target_orders` is the table after an ETL load.

The target table has a few intentional problems so we can practice finding them.

In [ ]:
source_orders = pd.DataFrame(
    {
        "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
        "customer_id": ["C001", "C002", "C003", "C004", "C005", "C001", "C006", "C007"],
        "order_date": [
            "2026-05-01",
            "2026-05-01",
            "2026-05-02",
            "2026-05-02",
            "2026-05-03",
            "2026-05-03",
            "2026-05-04",
            "2026-05-04",
        ],
        "status": ["complete", "complete", "refunded", "complete", "cancelled", "complete", "complete", "complete"],
        "gross_amount": [120.0, 75.0, 200.0, 45.0, 90.0, 130.0, 60.0, 40.0],
        "discount_amount": [0.0, 5.0, 20.0, 0.0, 90.0, 10.0, 0.0, 0.0],
        "net_amount": [120.0, 70.0, 180.0, 45.0, 0.0, 120.0, 60.0, 40.0],
        "updated_at": [
            "2026-05-01 09:00:00",
            "2026-05-01 09:05:00",
            "2026-05-02 10:00:00",
            "2026-05-02 10:10:00",
            "2026-05-03 11:00:00",
            "2026-05-03 11:15:00",
            "2026-05-04 12:00:00",
            "2026-05-04 12:05:00",
        ],
    }
)

source_orders["order_date"] = pd.to_datetime(source_orders["order_date"])
source_orders["updated_at"] = pd.to_datetime(source_orders["updated_at"])

source_orders

In [ ]:
target_orders = pd.DataFrame(
    {
        "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1008, 1009],
        "customer_id": ["C001", "C002", "C003", "C004", "C005", "C001", "C007", "C009"],
        "order_date": [
            "2026-05-01",
            "2026-05-01",
            "2026-05-02",
            "2026-05-02",
            "2026-05-03",
            "2026-05-03",
            "2026-05-04",
            "2026-05-04",
        ],
        "status": ["complete", "complete", "refunded", "complete", "cancelled", "complete", "complete", "complete"],
        "gross_amount": [120.0, 75.0, 200.0, 45.0, 90.0, 130.0, 40.0, 30.0],
        "discount_amount": [0.0, 5.0, 15.0, 0.0, 90.0, 10.0, 0.0, 0.0],
        "net_amount": [120.0, 70.0, 185.0, 45.0, 0.0, 120.0, 40.0, 30.0],
        "updated_at": [
            "2026-05-01 09:00:00",
            "2026-05-01 09:05:00",
            "2026-05-02 10:30:00",
            "2026-05-02 10:10:00",
            "2026-05-03 11:00:00",
            "2026-05-03 11:15:00",
            "2026-05-04 12:05:00",
            "2026-05-04 12:30:00",
        ],
    }
)

target_orders["order_date"] = pd.to_datetime(target_orders["order_date"])
target_orders["updated_at"] = pd.to_datetime(target_orders["updated_at"])

target_orders

## 3. Check row counts

Row count is usually the first reconciliation check.

Important: row count alone is not enough. Two tables can have the same number of rows and still contain different records.

In [ ]:
row_count_check = pd.DataFrame(
    [
        {
            "check_name": "row_count_match",
            "source_value": len(source_orders),
            "target_value": len(target_orders),
            "difference": len(target_orders) - len(source_orders),
            "passed": len(source_orders) == len(target_orders),
        }
    ]
)

row_count_check

## 4. Compare primary keys

The next check compares the primary key values. In this example, `order_id` identifies each order.

We want to know:

- Which source records are missing in the target?
- Which target records do not exist in the source?

In [ ]:
primary_key = "order_id"

source_keys = set(source_orders[primary_key])
target_keys = set(target_orders[primary_key])

missing_in_target = source_orders[~source_orders[primary_key].isin(target_keys)]
extra_in_target = target_orders[~target_orders[primary_key].isin(source_keys)]

print("Records from source that are missing in target:")
display(missing_in_target)

print("Records from target that do not exist in source:")
display(extra_in_target)

## 5. Reconcile aggregate metrics

After checking keys, we often compare important business metrics.

For orders, useful aggregate checks include:

- total `net_amount`
- total `gross_amount`
- counts by `status`
- totals by date or partition

In [ ]:
source_amounts = (
    source_orders.groupby("status", as_index=False)["net_amount"]
    .sum()
    .rename(columns={"net_amount": "source_net_amount"})
)

target_amounts = (
    target_orders.groupby("status", as_index=False)["net_amount"]
    .sum()
    .rename(columns={"net_amount": "target_net_amount"})
)

amount_reconciliation = source_amounts.merge(target_amounts, on="status", how="outer").fillna(0)
amount_reconciliation["difference"] = (
    amount_reconciliation["target_net_amount"] - amount_reconciliation["source_net_amount"]
)
amount_reconciliation["passed"] = amount_reconciliation["difference"].abs() <= 0.01

amount_reconciliation

## 6. Compare matched records column by column

If a key exists in both tables, the next question is whether the values match.

We will compare important columns for records that exist on both sides.

In [ ]:
matched_orders = source_orders.merge(
    target_orders,
    on=primary_key,
    how="inner",
    suffixes=("_source", "_target"),
)

columns_to_compare = [
    "customer_id",
    "order_date",
    "status",
    "gross_amount",
    "discount_amount",
    "net_amount",
]

differences = []

for column in columns_to_compare:
    source_column = f"{column}_source"
    target_column = f"{column}_target"
    mismatch_mask = matched_orders[source_column] != matched_orders[target_column]

    for _, row in matched_orders[mismatch_mask].iterrows():
        differences.append(
            {
                primary_key: row[primary_key],
                "column": column,
                "source_value": row[source_column],
                "target_value": row[target_column],
            }
        )

cell_level_differences = pd.DataFrame(differences)

cell_level_differences

## 7. Create a reconciliation summary

A good validation notebook should finish with a clear summary.

The summary should help someone quickly understand:

- which checks passed
- which checks failed
- how many records or values are affected
- where to investigate next

In [ ]:
total_source_net_amount = source_orders["net_amount"].sum()
total_target_net_amount = target_orders["net_amount"].sum()
total_net_amount_difference = total_target_net_amount - total_source_net_amount

summary_rows = [
    {
        "check_name": "row_count_match",
        "observed_value": len(target_orders),
        "expected_value": len(source_orders),
        "difference": len(target_orders) - len(source_orders),
        "passed": len(target_orders) == len(source_orders),
    },
    {
        "check_name": "no_missing_keys_in_target",
        "observed_value": len(missing_in_target),
        "expected_value": 0,
        "difference": len(missing_in_target),
        "passed": len(missing_in_target) == 0,
    },
    {
        "check_name": "no_extra_keys_in_target",
        "observed_value": len(extra_in_target),
        "expected_value": 0,
        "difference": len(extra_in_target),
        "passed": len(extra_in_target) == 0,
    },
    {
        "check_name": "total_net_amount_match",
        "observed_value": total_target_net_amount,
        "expected_value": total_source_net_amount,
        "difference": total_net_amount_difference,
        "passed": abs(total_net_amount_difference) <= 0.01,
    },
    {
        "check_name": "no_cell_level_differences",
        "observed_value": len(cell_level_differences),
        "expected_value": 0,
        "difference": len(cell_level_differences),
        "passed": len(cell_level_differences) == 0,
    },
]

reconciliation_summary = pd.DataFrame(summary_rows)
reconciliation_summary["status"] = reconciliation_summary["passed"].map({True: "PASS", False: "FAIL"})

reconciliation_summary

## 8. Interpret the result

A reconciliation summary should lead to investigation questions.

For this example, useful questions are:

- Why is `order_id = 1007` missing from the target?
- Why does `order_id = 1009` exist in the target but not the source?
- Why did the discount and net amount change for `order_id = 1003`?
- Did the pipeline use the right source snapshot or time window?
- Did the ETL logic filter, duplicate, or update records unexpectedly?

In [ ]:
failed_checks = reconciliation_summary[~reconciliation_summary["passed"]]

failed_checks

## Practice

Try these tasks:

1. Add a row count reconciliation by `status`.
2. Add a total `gross_amount` reconciliation.
3. Add a reconciliation by `order_date`.
4. Change the target data so all checks pass.
5. Write a short explanation of what failed and what you would check in the pipeline logs.